<a href="https://colab.research.google.com/github/Arif0000/llm-app/blob/main/Full_Enterprise_Self_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langgraph langchain-google-genai faiss-cpu python-dotenv



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.2
    Uninstalling langgraph-1.1.2:
      Successfully uninstalled langgraph-1.1.2
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12


In [2]:
!pip install -U langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
!pip install -U langchain-tavily

In [5]:
import os

os.environ["GOOGLE_API_KEY"] = "Enter your API keys"
os.environ["TAVILY_API_KEY"] = "enter your API keys"

In [6]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import json
import re

from dotenv import load_dotenv
load_dotenv()

# LangChain Gemini
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from langchain_tavily import TavilySearch

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [8]:
def parse_json(text: str):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    return {}


In [10]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.7 MB/s eta 0:00:00


In [11]:
docs = (
    PyPDFLoader("/content/Company_Policies.pdf").load()
    + PyPDFLoader("/content/Company_Profile.pdf").load()
    + PyPDFLoader("/content/Product_and_Pricing.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
).split_documents(docs)

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [12]:
class State(TypedDict):
    question: str
    retrieval_query: str
    rewrite_tries: int
    need_retrieval: bool
    docs: List[Document]
    relevant_docs: List[Document]
    context: str
    answer: str
    issup: str
    evidence: List[str]
    retries: int
    isuse: str
    use_reason: str
    web_query: str


In [13]:
def decide_retrieval(state: State):
    prompt = f"""
    Decide if retrieval is needed.
    Return JSON: {{ "should_retrieve": true/false }}

    Question: {state['question']}
    """
    res = llm.invoke(prompt)
    parsed = parse_json(res.content)
    return {"need_retrieval": parsed.get("should_retrieve", True)}

def route_after_decide(state):
    return "retrieve" if state["need_retrieval"] else "generate_direct"


In [14]:
def generate_direct(state: State):
    res = llm.invoke(state["question"])
    return {"answer": res.content}

In [15]:
def retrieve(state: State):
    query = state.get("retrieval_query") or state["question"]
    return {"docs": retriever.invoke(query)}

In [16]:
def is_relevant(state: State):
    relevant = []

    for doc in state["docs"]:
        prompt = f"""
        Is this document relevant?

        Return JSON: {{ "is_relevant": true/false }}

        Q: {state['question']}
        Doc: {doc.page_content}
        """

        res = llm.invoke(prompt)
        parsed = parse_json(res.content)

        if parsed.get("is_relevant", False):
            relevant.append(doc)

    return {"relevant_docs": relevant}

def route_after_relevance(state):
    return "generate_from_context" if state["relevant_docs"] else "rewrite_question"

In [17]:
def generate_from_context(state: State):
    context = "\n\n".join([d.page_content for d in state["relevant_docs"]])

    prompt = f"""
    Answer ONLY from context.

    Question: {state['question']}

    Context:
    {context}
    """

    res = llm.invoke(prompt)

    return {"answer": res.content, "context": context}


In [18]:
def is_sup(state: State):
    prompt = f"""
    Check support.

    Return JSON:
    {{
      "issup": "fully_supported | partially_supported | no_support",
      "evidence": ["text"]
    }}

    Answer: {state['answer']}
    Context: {state['context']}
    """

    res = llm.invoke(prompt)
    parsed = parse_json(res.content)

    return {
        "issup": parsed.get("issup", "no_support"),
        "evidence": parsed.get("evidence", [])
    }

def route_after_sup(state):
    if state["issup"] == "fully_supported":
        return "is_use"
    if state["retries"] >= 3:
        return "is_use"
    return "revise_answer"


In [19]:
def revise_answer(state: State):
    res = llm.invoke(
        f"Rewrite strictly from context:\n{state['context']}"
    )

    return {
        "answer": res.content,
        "retries": state["retries"] + 1
    }

In [20]:
def is_use(state: State):
    prompt = f"""
    Return JSON:
    {{ "isuse": "useful | not_useful", "reason": "short" }}

    Q: {state['question']}
    A: {state['answer']}
    """

    res = llm.invoke(prompt)
    parsed = parse_json(res.content)

    return {
        "isuse": parsed.get("isuse", "not_useful"),
        "use_reason": parsed.get("reason", "")
    }

def route_after_use(state):
    if state["isuse"] == "useful":
        return END
    if state["rewrite_tries"] >= 2:
        return "web_fallback"
    return "rewrite_question"


In [21]:
def rewrite_question(state: State):
    res = llm.invoke(
        f"Rewrite for retrieval:\n{state['question']}"
    )

    return {
        "retrieval_query": res.content,
        "rewrite_tries": state["rewrite_tries"] + 1
    }

In [22]:
tavily = TavilySearch(max_results=5)

def web_fallback(state: State):
    results = tavily.invoke({"query": state["question"]})

    docs = [
        Document(page_content=r["content"], metadata={"source": "web"})
        for r in results
    ]

    return {"docs": docs}

In [23]:
g = StateGraph(State)

g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)
g.add_node("is_relevant", is_relevant)
g.add_node("generate_from_context", generate_from_context)
g.add_node("is_sup", is_sup)
g.add_node("revise_answer", revise_answer)
g.add_node("is_use", is_use)
g.add_node("rewrite_question", rewrite_question)
g.add_node("web_fallback", web_fallback)

g.add_edge(START, "decide_retrieval")

g.add_conditional_edges("decide_retrieval", route_after_decide, {
    "retrieve": "retrieve",
    "generate_direct": "generate_direct"
})

g.add_edge("generate_direct", END)
g.add_edge("retrieve", "is_relevant")

g.add_conditional_edges("is_relevant", route_after_relevance, {
    "generate_from_context": "generate_from_context",
    "rewrite_question": "rewrite_question"
})

g.add_edge("generate_from_context", "is_sup")

g.add_conditional_edges("is_sup", route_after_sup, {
    "is_use": "is_use",
    "revise_answer": "revise_answer"
})

g.add_edge("revise_answer", "is_sup")

g.add_conditional_edges("is_use", route_after_use, {
    END: END,
    "rewrite_question": "rewrite_question",
    "web_fallback": "web_fallback"
})

g.add_edge("rewrite_question", "retrieve")
g.add_edge("web_fallback", "is_relevant")

app = g.compile()

In [24]:
result = app.invoke({
    "question": "Who is the CEO of NexaAI?",
    "retrieval_query": "",
    "rewrite_tries": 0,
    "docs": [],
    "relevant_docs": [],
    "context": "",
    "answer": "",
    "issup": "",
    "evidence": [],
    "retries": 0,
    "isuse": "not_useful",
    "use_reason": "",
    "web_query": ""
})

print(result["answer"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-pro' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\nPlease retry in 57.534196248s.', 'status': 'RESOURCE_EXHAUST

In [25]:
print("\n===== DEBUG =====\n")

print("Need Retrieval:", result.get("need_retrieval"))
print("Docs Retrieved:", len(result.get("docs", [])))
print("Relevant Docs:", len(result.get("relevant_docs", [])))

print("\nIsSUP:", result.get("issup"))
print("Evidence:", result.get("evidence"))

print("\nIsUSE:", result.get("isuse"))
print("Reason:", result.get("use_reason"))

print("\nFinal Answer:\n", result.get("answer"))


===== DEBUG =====



NameError: name 'result' is not defined